# 03 - Interpretacion Visual e Importancia de Variables en Arboles de Regresion

## Curso de Machine Learning: Modelos de Regresion

Una de las ventajas mas significativas de los arboles de decision frente a otros modelos (redes neuronales, SVM con kernel, etc.) es su **interpretabilidad intrinseca**. Un arbol de decision es, esencialmente, un diagrama de flujo que puede ser leido y comprendido por cualquier profesional del dominio, sin necesidad de conocimientos avanzados en estadistica o algebra lineal.

En este cuaderno profundizaremos en las herramientas de interpretacion y explicabilidad que los arboles de regresion ofrecen, y las contrastaremos con la interpretacion basada en coeficientes de los modelos lineales.

---

### Contenidos
1. Interpretacion de la Estructura del Arbol como Sistema de Reglas
2. Visualizacion Completa con `plot_tree` y Personalizacion
3. Representacion Textual con `export_text` para Reportes
4. Importancia de Variables (Feature Importance) basada en Impureza
5. Limitaciones de la Importancia basada en Impureza
6. Importancia por Permutacion: Metodo Independiente del Modelo
7. Comparativa: Importancia de Variables en Arbol vs. Coeficientes de Ridge
8. Dependencia Parcial (Partial Dependence): Efecto Marginal de una Variable
9. Prediccion Individual: Descomposicion de la Ruta del Arbol
10. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, plot_tree, export_text
from sklearn.linear_model import Ridge
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import mean_squared_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)

# Carga de datos
diabetes = load_diabetes(scaled=False, as_frame=True)
X = diabetes.frame.drop(columns=['target'])
y = diabetes.frame['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Arbol podado (profundidad moderada)
tree_reg = DecisionTreeRegressor(max_depth=4, min_samples_leaf=10, random_state=42)
tree_reg.fit(X_train, y_train)

print(f"Arbol entrenado: profundidad={tree_reg.get_depth()}, hojas={tree_reg.get_n_leaves()}")
print(f"R2 Train: {r2_score(y_train, tree_reg.predict(X_train)):.4f}")
print(f"R2 Test:  {r2_score(y_test, tree_reg.predict(X_test)):.4f}")


## 1. Interpretacion de la Estructura del Arbol como Sistema de Reglas

Cada ruta desde el nodo raiz hasta una hoja terminal define una **regla de decision conjuntiva** (AND). Por ejemplo:

```
SI bmi <= 25.35
   Y s5 <= 4.55
   Y age <= 50
ENTONCES prediccion = 85.7
```

El arbol completo es equivalente a un sistema de reglas **mutuamente excluyentes y exhaustivas**: toda observacion cae exactamente en una hoja, y toda observacion tiene exactamente una prediccion.

Esta transparencia convierte a los arboles en candidatos ideales para dominios donde la auditabilidad del modelo es un requisito regulatorio (medicina, finanzas, seguros).


## 2. Visualizacion Completa con `plot_tree`

La funcion `sklearn.tree.plot_tree` genera una representacion grafica del arbol donde cada nodo muestra:
- La condicion de division (`feature <= threshold`)
- La metrica de impureza del nodo (`squared_error`)
- El numero de muestras (`samples`)
- El valor predicho (`value`, la media de las etiquetas)
- El color del nodo (mas intenso = menor impureza o prediccion mas extrema)


In [ ]:
# Visualizacion con plot_tree
plt.figure(figsize=(22, 12))
plot_tree(tree_reg, 
          feature_names=X.columns.tolist(), 
          filled=True, 
          rounded=True,
          fontsize=8, 
          proportion=True,
          precision=1,
          impurity=True)
plt.title('Arbol de Regresion (max_depth=4, min_samples_leaf=10) - Dataset Diabetes',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 3. Representacion Textual con `export_text`

Para documentacion tecnica, auditorias o integracion en pipelines de produccion, la representacion textual permite extraer las reglas como texto plano.


In [ ]:
# Representacion textual completa
texto = export_text(tree_reg, feature_names=X.columns.tolist(), decimals=2, show_weights=True)
print("=" * 70)
print("ARBOL DE DECISION - REGLAS COMPLETAS")
print("=" * 70)
print(texto)


## 4. Importancia de Variables Basada en Impureza (MDI)

Los arboles de decision calculan una medida natural de importancia para cada variable predictora denominada **Mean Decrease in Impurity (MDI)**. Para una variable $x_j$, su importancia se calcula como la suma ponderada de las reducciones de impureza a traves de todos los nodos del arbol donde $x_j$ fue la variable de division:

$$\text{Importancia}(x_j) = \sum_{\text{nodo } n \text{ usa } x_j} \frac{n_\text{samples}(n)}{n_\text{total}} \cdot \Delta I(n)$$

Las importancias se normalizan para sumar 1.0. Esta medida esta disponible directamente en el atributo `feature_importances_` del modelo entrenado.


In [ ]:
# Importancia de variables basada en impureza (MDI)
importancias_mdi = pd.DataFrame({
    'Variable': X.columns,
    'Importancia_MDI': tree_reg.feature_importances_
}).sort_values('Importancia_MDI', ascending=True)

plt.figure(figsize=(10, 6))
colores = plt.cm.viridis(np.linspace(0.2, 0.9, len(importancias_mdi)))
plt.barh(importancias_mdi['Variable'], importancias_mdi['Importancia_MDI'], 
         color=colores, edgecolor='#333333', linewidth=0.5)
plt.xlabel('Importancia (MDI - Reduccion de Impureza)', fontsize=11, fontweight='bold')
plt.title('Importancia de Variables - Arbol de Regresion (MDI)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Top 5 variables
print("Top 5 Variables por Importancia (MDI):")
print(importancias_mdi.sort_values('Importancia_MDI', ascending=False).head().to_string(index=False))


## 5. Limitaciones de la Importancia Basada en Impureza

La importancia MDI tiene limitaciones bien documentadas:

1. **Sesgo hacia variables de alta cardinalidad**: Variables con muchos valores unicos (ej. variables continuas) tienden a tener importancias infladas porque ofrecen mas puntos de corte potenciales.
2. **Variables correlacionadas**: Cuando dos variables estan altamente correlacionadas, el arbol puede usar una u otra indistintamente. La importancia se reparte de forma arbitraria entre ambas, subestimando la importancia real de cada una individualmente.
3. **No captura la direccion del efecto**: A diferencia de los coeficientes lineales ($\beta_j > 0$ indica efecto positivo), la importancia MDI solo indica **cuanto** contribuye una variable, pero no la **direccion** de su efecto.
4. **Dependencia del arbol especifico**: Cambios menores en los datos pueden producir arboles con estructuras radicalmente diferentes y, por tanto, importancias muy distintas (alta inestabilidad).


## 6. Importancia por Permutacion: Metodo Independiente del Modelo

La **importancia por permutacion** (Breiman, 2001) es un metodo **model-agnostic** que evalua la importancia de una variable midiendo cuanto se degrada el rendimiento del modelo cuando se permutan aleatoriamente los valores de esa variable, rompiendo su relacion con la variable objetivo.

$$\text{Importancia}_{perm}(x_j) = S(y, \hat{f}(X)) - \frac{1}{K} \sum_{k=1}^{K} S(y, \hat{f}(X_{\pi_k(j)}))$$

Donde $S$ es la metrica de evaluacion (ej. $R^2$), $X_{\pi_k(j)}$ es la matriz de datos con la columna $j$ permutada, y $K$ es el numero de repeticiones de la permutacion.


In [ ]:
# Importancia por permutacion (evaluada en TEST)
perm_imp = permutation_importance(tree_reg, X_test, y_test, 
                                   n_repeats=30, random_state=42, scoring='r2')

importancias_perm = pd.DataFrame({
    'Variable': X.columns,
    'Importancia_Perm': perm_imp.importances_mean,
    'Std': perm_imp.importances_std
}).sort_values('Importancia_Perm', ascending=True)

plt.figure(figsize=(10, 6))
colores = plt.cm.magma(np.linspace(0.2, 0.85, len(importancias_perm)))
plt.barh(importancias_perm['Variable'], importancias_perm['Importancia_Perm'],
         xerr=importancias_perm['Std'], color=colores, edgecolor='#333333', 
         linewidth=0.5, capsize=3)
plt.xlabel('Importancia por Permutacion (caida en $R^2$)', fontsize=11, fontweight='bold')
plt.title('Importancia de Variables - Permutacion (evaluada en Test)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("Top 5 Variables por Importancia de Permutacion:")
print(importancias_perm.sort_values('Importancia_Perm', ascending=False).head().round(4).to_string(index=False))


## 7. Comparativa: Importancia en Arbol vs. Coeficientes de Ridge

Comparamos la importancia derivada del arbol con los coeficientes estandarizados del modelo Ridge para contrastar dos paradigmas de interpretacion distintos.


In [ ]:
# Ridge para comparacion (con datos estandarizados)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

pipe_ridge = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=1.0))
])
pipe_ridge.fit(X_train, y_train)

coefs_ridge = pipe_ridge.named_steps['ridge'].coef_

# Comparativa visual
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel 1: Importancia del Arbol
importancias_mdi_sorted = importancias_mdi.sort_values('Importancia_MDI', ascending=True)
axes[0].barh(importancias_mdi_sorted['Variable'], importancias_mdi_sorted['Importancia_MDI'], 
             color='#2b7bba', edgecolor='#333333', linewidth=0.5)
axes[0].set_xlabel('Importancia (MDI)', fontsize=11, fontweight='bold')
axes[0].set_title('Arbol de Decision: Importancia de Variables', fontsize=12, fontweight='bold')

# Panel 2: Coeficientes Ridge
coefs_df = pd.DataFrame({
    'Variable': X.columns,
    'Coef_Ridge': coefs_ridge
}).sort_values('Coef_Ridge')
colors_coef = ['#d62728' if c < 0 else '#2ca02c' for c in coefs_df['Coef_Ridge']]
axes[1].barh(coefs_df['Variable'], coefs_df['Coef_Ridge'], color=colors_coef, 
             edgecolor='#333333', linewidth=0.5)
axes[1].set_xlabel('Coeficiente Estandarizado (Ridge)', fontsize=11, fontweight='bold')
axes[1].set_title('Ridge: Coeficientes (Direccion + Magnitud)', fontsize=12, fontweight='bold')
axes[1].axvline(0, color='black', linewidth=0.8)

plt.suptitle('Dos Paradigmas de Interpretacion: Arbol (MDI) vs. Modelo Lineal (Coeficientes)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 8. Dependencia Parcial (Partial Dependence)

Los **Graficos de Dependencia Parcial (PDP)** muestran el efecto marginal de una o dos variables sobre la prediccion del modelo, promediando sobre los valores de las demas variables. Esto permite recuperar la **direccion del efecto** que la importancia MDI no proporciona.

Para un arbol de regresion, la funcion de dependencia parcial es una funcion escalonada (piecewise constant).


In [ ]:
# Graficos de Dependencia Parcial para las 4 variables mas importantes
top_4_vars = importancias_mdi.sort_values('Importancia_MDI', ascending=False).head(4)['Variable'].tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
display = PartialDependenceDisplay.from_estimator(
    tree_reg, X_train, features=top_4_vars,
    kind='average', grid_resolution=100,
    ax=axes.ravel()
)

fig.suptitle('Dependencia Parcial: Efecto Marginal de las Variables mas Importantes',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 9. Prediccion Individual: Descomposicion de la Ruta del Arbol

Para una prediccion individual, podemos trazar la ruta completa que la observacion sigue desde el nodo raiz hasta la hoja terminal. Esta transparencia total es conocida como **path-based explanation** y es una de las fortalezas principales de los arboles de decision.


In [ ]:
# Descomposicion de la prediccion para una observacion individual
idx_ejemplo = 0
x_individual = X_test.iloc[[idx_ejemplo]]
y_real = y_test.iloc[idx_ejemplo]
y_pred = tree_reg.predict(x_individual)[0]

# Ruta del nodo
nodo_indicator = tree_reg.decision_path(x_individual)
nodo_ids = nodo_indicator.indices

# Informacion del arbol
tree = tree_reg.tree_
feature_names = X.columns.tolist()

print("=" * 70)
print(f"PREDICCION INDIVIDUAL (Observacion #{idx_ejemplo})")
print("=" * 70)
print(f"  Valor Real:    {y_real:.1f}")
print(f"  Prediccion:    {y_pred:.1f}")
print(f"  Error:         {abs(y_real - y_pred):.1f}")
print()
print("Ruta del Arbol (Nodos Visitados):")
print("-" * 70)

for i, nodo_id in enumerate(nodo_ids):
    if tree.feature[nodo_id] >= 0:  # Nodo interno
        feature_name = feature_names[tree.feature[nodo_id]]
        threshold = tree.threshold[nodo_id]
        valor_obs = x_individual[feature_name].values[0]
        direccion = "<=" if valor_obs <= threshold else ">"
        print(f"  Nodo {nodo_id}: {feature_name} = {valor_obs:.2f} {direccion} {threshold:.2f} "
              f"-> {'Izquierda' if direccion == '<=' else 'Derecha'}")
    else:
        print(f"  Hoja {nodo_id}: Prediccion = {tree.value[nodo_id][0][0]:.1f} "
              f"(basada en {tree.n_node_samples[nodo_id]} muestras de entrenamiento)")


## 10. Conclusiones

### Puntos Clave:
1. **Transparencia Total**: Los arboles de decision son sistemas de reglas legibles. Cada prediccion puede ser trazada paso a paso desde la raiz hasta la hoja, lo que los hace ideales para dominios con requisitos de auditabilidad.
2. **Importancia MDI**: Es rapida y gratuita (se calcula durante el entrenamiento) pero sufre de sesgos hacia variables de alta cardinalidad y no refleja la direccion del efecto.
3. **Importancia por Permutacion**: Es model-agnostic y mas robusta, pero computacionalmente mas costosa. Evalua la contribucion real de cada variable al rendimiento predictivo.
4. **Dependencia Parcial**: Recupera la forma funcional del efecto marginal de cada variable, complementando la informacion de importancia con la direccion del efecto.
5. **Coeficientes vs. Importancia**: En modelos lineales, los coeficientes proporcionan magnitud y direccion simultaneamente. En arboles, se requiere combinar importancia MDI + PDP para obtener ambas perspectivas.

En el siguiente cuaderno, abordaremos los **ejercicios practicos** que consolidaran todos los conceptos estudiados en este modulo.
